# 🌌 Aurora · Wan video generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subash1327/aurora/blob/main/notebooks/aurora_video_wan.ipynb)

Text-to-video and image-to-video with [Wan 2.1 / 2.2](https://huggingface.co/Wan-AI) through 🤗 diffusers.

| Model | Mode | Runs on |
|---|---|---|
| **Wan2.1-T2V-1.3B** (default) | text → video, 480p | free **T4** |
| **Wan2.2-TI2V-5B** | text/image → video, 720p 24 fps | T4 (4-bit, slow) · **L4** recommended |
| Wan2.1-T2V-14B, Wan2.1-I2V-14B-480P/720P | text / image → video | **A100** |
| Wan2.2-T2V-A14B, Wan2.2-I2V-A14B (MoE) | text / image → video | A100 80 GB (40 GB with offload) |

**Steps:** run 1 → 2 → 3 (load model) → 3b (i2v: upload a start image from your computer, optional) → 4 (generate). Re-run 4 as often as you like.

> **Content policy:** no sexual content involving minors (prompts with child/teen terms are refused automatically) and no sexual or non-consensual content of real, identifiable people. See the README.

In [ ]:
#@title 1. Install Aurora + dependencies (≈2 min)
AURORA_REF = "main"  #@param {type:"string"}
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!pip install -q -U -r https://raw.githubusercontent.com/subash1327/aurora/{AURORA_REF}/requirements.txt
!pip install -q -U --no-deps --force-reinstall "git+https://github.com/subash1327/aurora@{AURORA_REF}"
print("Installed. If Colab asks to restart the session, restart and continue from step 2.")

In [ ]:
#@title 2. Hugging Face login (Colab secret `HF_TOKEN`) + GPU check
#@markdown Add `HF_TOKEN` in the 🔑 **Secrets** sidebar and enable notebook access. Needed for gated models (e.g. FLUX).
from aurora.device import get_device_info, hf_login_from_colab_secret

hf_login_from_colab_secret("HF_TOKEN")
INFO = get_device_info()
print(INFO.summary())
if INFO.device != "cuda":
    print("⚠️  No GPU detected: Runtime > Change runtime type > T4 GPU (or L4 / A100).")

In [ ]:
#@title 3. Load a Wan model
MODEL = "auto"  #@param ["auto", "Wan2.1-T2V-1.3B", "Wan2.2-TI2V-5B", "Wan2.1-T2V-14B", "Wan2.1-I2V-14B-480P", "Wan2.1-I2V-14B-720P", "Wan2.2-T2V-A14B", "Wan2.2-I2V-A14B"]
MODE = "t2v"  #@param ["t2v", "i2v"]
#@markdown `auto` = Wan2.1-T2V-1.3B (t2v) / Wan2.2-TI2V-5B (i2v) on T4/L4, 14B models on A100.
DTYPE = "auto"  #@param ["auto", "fp16", "bf16", "fp32"]
#@markdown `auto`: bf16 on L4/A100, fp16 on T4 (no fast bf16).
OFFLOAD = "auto"  #@param ["auto", "none", "model", "sequential"]
QUANTIZE = "auto"  #@param ["auto", "none", "text_encoder_8bit", "8bit", "4bit"]
#@markdown `auto` on T4: 8-bit text encoder (+ 4-bit transformer for 5B+). `sequential` offload = lowest VRAM, slowest.
VAE_TILING = True  #@param {type:"boolean"}

from aurora.device import free_memory
from aurora.video import WanGenerator

_key = (MODEL, MODE, DTYPE, OFFLOAD, QUANTIZE, VAE_TILING)
if globals().get("GEN") is None or globals().get("_GEN_KEY") != _key:
    if globals().get("GEN") is not None:
        GEN.unload()
        GEN = None
        free_memory()
    GEN = WanGenerator(model=MODEL, mode=MODE, dtype=DTYPE, offload=OFFLOAD, quantize=QUANTIZE, vae_tiling=VAE_TILING)
    _GEN_KEY = _key
s = GEN.spec
print(f"Ready: {s.key} [{GEN.mode}] defaults {s.height}x{s.width}, {s.num_frames} frames @ {s.fps} fps, "
      f"{s.steps} steps, guidance {s.guidance}. {s.notes}")

In [ ]:
#@title 3b. (i2v only) Upload a start image from your computer
UPLOAD_METHOD = "file picker"  #@param ["file picker", "drag & drop widget"]
#@markdown `file picker` opens your computer's file dialog. `drag & drop widget` shows a drop zone; drop the file, then run step 4.
CLEAR_PREVIOUS = True  #@param {type:"boolean"}
#@markdown An uploaded image takes precedence over `INPUT_IMAGE` in step 4 (untick `USE_UPLOADED` there to ignore it).
from aurora.inputs import upload

if CLEAR_PREVIOUS and globals().get("IMAGE_UPLOADS") is not None:
    IMAGE_UPLOADS.clear()
IMAGE_UPLOADS = upload("image", UPLOAD_METHOD, uploader=globals().get("IMAGE_UPLOADS"))

In [ ]:
#@title 4. Generate a video
PROMPT = "A cinematic drone shot gliding over snowy mountains at sunrise, golden light, volumetric fog, ultra detailed"  #@param {type:"string"}
NEGATIVE_PROMPT = "default"  #@param {type:"string"}
#@markdown `default` = Wan's recommended negative prompt. Leave blank for none.
RESOLUTION = "model default"  #@param ["model default", "480x832 (landscape)", "832x480 (portrait)", "480x480", "704x1280 (landscape 720p)", "1280x704 (portrait 720p)", "720x1280 (landscape)", "1280x720 (portrait)"]
NUM_FRAMES = 81  #@param {type:"slider", min:17, max:121, step:4}
#@markdown Frames must be 4k+1 (snapped automatically). Wan2.1: 81 @ 16 fps ≈ 5 s. Wan2.2-5B: 121 @ 24 fps ≈ 5 s.
FPS = 0  #@param {type:"slider", min:0, max:30, step:1}
#@markdown `0` = model default (16 for Wan 2.1, 24 for Wan2.2-TI2V-5B).
STEPS = 30  #@param {type:"slider", min:4, max:60, step:1}
GUIDANCE = 5.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
SEED = -1  #@param {type:"integer"}
#@markdown `-1` = random seed.
INPUT_IMAGE = ""  #@param {type:"string"}
#@markdown **i2v only:** image URL, `/content/...` path or Drive path (`/content/drive/MyDrive/...`). An image uploaded in step 3b wins; if nothing is set, a file picker opens.
USE_UPLOADED = True  #@param {type:"boolean"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

import time

from aurora.device import vram_report
from aurora.outputs import copy_to_drive, save_video, show_video
from aurora.video import DEFAULT_NEGATIVE_PROMPT

neg = DEFAULT_NEGATIVE_PROMPT if NEGATIVE_PROMPT.strip() == "default" else NEGATIVE_PROMPT
h = w = None
if RESOLUTION != "model default":
    h, w = (int(x) for x in RESOLUTION.split()[0].split("x"))

image = None
if GEN.mode == "i2v":
    from aurora.inputs import resolve_image

    image = resolve_image(INPUT_IMAGE, uploader=globals().get("IMAGE_UPLOADS") if USE_UPLOADED else None)
    if image is None:
        raise ValueError("Image-to-video needs a start image: upload one in step 3b or set INPUT_IMAGE.")

t0 = time.time()
frames, params = GEN.generate(PROMPT, negative_prompt=neg, image=image, height=h, width=w,
                              num_frames=NUM_FRAMES, steps=STEPS, guidance=GUIDANCE, seed=SEED)
fps = FPS or GEN.spec.fps
params["fps"] = fps
path = save_video(frames, PROMPT, fps=fps, seed=params["seed"], metadata=params)
print(f"Done in {time.time() - t0:.0f}s | seed={params['seed']} | {params['height']}x{params['width']} x {params['num_frames']} frames | {vram_report()}")
print(f"Saved: {path}")
show_video(path)
if SAVE_TO_DRIVE:
    copy_to_drive(path, "Aurora/videos")

In [ ]:
#@title (Optional) Download the last video
from google.colab import files

files.download(str(path))

## Tips & troubleshooting

* **CUDA out of memory**: lower `NUM_FRAMES` (e.g. 49) or resolution, set `OFFLOAD = "sequential"`, or `QUANTIZE = "4bit"`. Re-run step 3 after changing load settings. *Runtime > Restart session* clears fragmented VRAM.
* **Session crashed / RAM full while loading** (free Colab has ~12.7 GB system RAM): keep `QUANTIZE = "auto"` (8-bit text encoder), use the 1.3B model, or switch to a High-RAM / L4 runtime.
* **Black or noisy video on T4**: fp16 can overflow on some checkpoints. Try `DTYPE = "fp32"` with `OFFLOAD = "sequential"` (slow), or use an L4/A100 with bf16.
* **Slow**: the 1.3B model on a T4 takes several minutes per 5 s clip. Fewer steps (20-25) is a good trade-off.
* Wan 2.2-TI2V-5B is trained at 720p (704x1280); lower resolutions work but look softer.
